## Face Beauty Workout Website Scraping

1. Sent http endpoint request for json response of beauty workout website contents
2. The Ollama model is used to scrap the website contents

### We will start by calling OpenAI again - embedding local llama3.2 model

In [ ]:
#importing python client packages
import os
from dotenv import load_dotenv


load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API Key is found. PLease head over to the troubleshooting notebook in the setup folder")
elif not api_key.startswith("sk-proj-"):
    print("An API Key is found but doesn't start with 'sk-proj'; please check you're using right key - see troubleshooting nootbook")
else:
    print("An API Key is found and looks good so far")

## Types of Prompt

In [ ]:
# Define our system prompt"

system_prompt = """
You are a personalised assistant that thoroughly read and understand the contents of a website,
and provides brief ideas about different types of Face Toned mathods; Face Gym, ignoring text that might be navigation related.
The output should be in a simple and easy to understand format, wrapped in markdown instead of code blocks - don't include any code blocks in your response.
"""

# Define our user prompt

user_prompt_prefix = """
Here are the contents of the website and provide a brief summary of different types of Face Toned methods.
It includes facial exercises, it's necessity, modern beauty.
Explain facial fitness and beauty routine, frequency, and results in a separate table
"""

# Make the call to the model

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

## First Method: using http endpoint to scrap the website content

1. Using header to wrap http requests for API 
2. json file request that show results of scrapped website contents
2. Here ollama is locally installed, no need of 'api_key', referring the llama3.2 model

And, here is an endpoint that describes the processes...

In [ ]:
from scraper import fetch_website_contents
import requests

def summarize_json(url):
    face_tone_website = fetch_website_contents(url)

    headers = {"Authorization": "bearer 'ollama_api_key'", "Content-Type": "application/json"}

    payload = {
        "model": "llama3.2",
        "messages": messages_for(face_tone_website)
    }

    response = requests.post(
        "http://localhost:11434/v1/chat/completions",
        headers=headers,
        json=payload
    )

    response.json()
    return response.json()["choices"][0]["message"]["content"]

In [ ]:
summarize_json("https://facetoned.com/face-gym-method/")

In [ ]:
# A function to display this nicely in the output, using markdown
from IPython.display import Markdown, display

def display_summary(url):
    summary = summarize_json(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://facetoned.com/face-gym-method/")

## Second Method: Using OpenAI package to scrap Face Toned website

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!

In [ ]:
#check Ollama is running!
requests.get("http://localhost:11434").content

In [ ]:
#Download llama 3.2 from meta
!ollama pull llama3.2

In [ ]:
#Let's call OpenAI: Python client Library

from openai import OpenAI

openai = OpenAI(base_url="http://localhost:11434/v1", api_key= "ollama")

def summarize(url):
    face_tone_website = fetch_website_contents(url)

    response = openai.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(face_tone_website)
    )

    return response.choices[0].message.content

In [ ]:
summarize("https://facetoned.com/face-gym-method/")

In [ ]:
# A function to display this nicely in the output, using markdown
from IPython.display import Markdown, display

def display_summary(url):
    summary = summarize_json(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://facetoned.com/face-gym-method/")